In [1]:
# Build SoCal BASE file (run once) 
import pandas as pd
from pathlib import Path

PROC = Path("../../1_DATA/processed")
SOCAL_BASE = PROC / "socal_kelp_sst_base.csv"

SOCAL_SST_MONTHLY = PROC / "oisst_socalv1_bbox_monthly.csv"
SOCAL_KELP_Q      = PROC / "kelp_timeseries_socalv1_bbox_quarterly.csv"

def load_timeseries_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    for col in ["time", "date", "datetime", "timestamp"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
            return df.set_index(col).sort_index()
    return pd.read_csv(path, index_col=0, parse_dates=True).sort_index()

# build SoCal quarterly SST features
sst_m = pd.read_csv(SOCAL_SST_MONTHLY, parse_dates=["time"]).set_index("time").sort_index()
q = sst_m.resample("QS")
feat = pd.DataFrame({
    "sst_q_mean": q["sst"].mean(),
    "sstanom_q_mean": q["sst_anom"].mean(),
    "sstanom_q_max": q["sst_anom"].max(),
})
feat["sstanom_q_mean_lag1"] = feat["sstanom_q_mean"].shift(1)

# load SoCal kelp quarterly and align to quarter-start for join
kelp = load_timeseries_csv(SOCAL_KELP_Q)
kelp_times = pd.DatetimeIndex(kelp.index)
kelp_qstart = kelp_times.to_period("Q").to_timestamp(how="start")

feat_aligned = feat.reindex(kelp_qstart)
feat_aligned.index = kelp_times

base = kelp.join(feat_aligned, how="left")
base.to_csv(SOCAL_BASE, index=True)

print("Wrote BASE:", SOCAL_BASE.resolve())
print("Columns:", list(base.columns))
print("Rows:", len(base), "Time:", base.index.min(), "->", base.index.max())

Wrote BASE: /Users/tonylin/Documents/kelp_project/1_DATA/processed/socal_kelp_sst_base.csv
Columns: ['kelp_area', 'coverage', 'kelp_smooth', 'coverage_frac', 'sst_q_mean', 'sstanom_q_mean', 'sstanom_q_max', 'sstanom_q_mean_lag1']
Rows: 167 Time: 1984-02-15 00:00:00 -> 2025-08-15 00:00:00


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PROC = Path("../../1_DATA/processed")
SOCAL_BASE = PROC / "socal_kelp_sst_base.csv"
SOCAL_LABELED = PROC / "socal_kelp_sst_labeled.csv"

d = pd.read_csv(SOCAL_BASE, index_col=0, parse_dates=True).sort_index()

# choose kelp variable to define "low kelp state"
# (use what you actually trust most for SoCal)
if "coverage_frac" in d.columns:
    k = d["coverage_frac"].astype(float)
    source = "coverage_frac"
elif "kelp_smooth" in d.columns:
    k = d["kelp_smooth"].astype(float)
    source = "kelp_smooth"
else:
    raise ValueError(f"No kelp column found. Columns: {list(d.columns)}")

# z-score then smooth (4 quarters ~ 1 year)
z = (k - k.mean()) / k.std()
z_s = z.rolling(4, min_periods=3).mean()

# OPTION: state-based threshold (tune THRESH)
THRESH = -1.0
low = (z_s < THRESH)

# require persistence: at least 2 consecutive quarters low
low_state = low & (low.shift(1).fillna(False) | low.shift(-1).fillna(False))

d["suppressed"] = low_state.astype(int)

d.to_csv(SOCAL_LABELED, index=True)
print("Wrote LABELED:", SOCAL_LABELED.resolve())
print("Label built from:", source, "| THRESH:", THRESH, "| smoothing=4Q | persistence>=2Q")
print("Suppressed counts:", d["suppressed"].value_counts().to_dict())
print("Columns now include suppressed?", "suppressed" in d.columns)

Wrote LABELED: /Users/tonylin/Documents/kelp_project/1_DATA/processed/socal_kelp_sst_labeled.csv
Label built from: coverage_frac | THRESH: -1.0 | smoothing=4Q | persistence>=2Q
Suppressed counts: {0: 159, 1: 8}
Columns now include suppressed? True


/var/folders/61/3wm_gk5j5jvd7cv3mv31_8940000gn/T/ipykernel_55165/2751537516.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  low_state = low & (low.shift(1).fillna(False) | low.shift(-1).fillna(False))


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score

DATA_PATH = Path("../../1_DATA/processed/socal_kelp_sst_labeled.csv")
d = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True).sort_index()

# Align to quarter-start
d.index = d.index.to_period("Q").to_timestamp(how="start")

# Check required columns
required = ["suppressed", "sstanom_q_max"]
missing = [c for c in required if c not in d.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}\nColumns in file: {list(d.columns)}")

y = d["suppressed"].astype(int)
x0 = d["sstanom_q_max"]

print("Loaded:", DATA_PATH.resolve())
print("Rows:", len(d), "Time:", d.index.min(), "->", d.index.max())
print("Suppressed counts:", y.value_counts().to_dict())

# Compute AUC + mean differences by lag
MAX_LAG = 8
rows = []
for L in range(0, MAX_LAG + 1):
    x = x0.shift(L)  # SST earlier -> suppressed later
    tmp = pd.DataFrame({"x": x, "y": y}).dropna()
    if tmp["y"].nunique() < 2:
        continue
    auc = roc_auc_score(tmp["y"], tmp["x"])
    mean0 = tmp[tmp["y"] == 0]["x"].mean()
    mean1 = tmp[tmp["y"] == 1]["x"].mean()
    rows.append((L, len(tmp), auc, mean0, mean1, mean1-mean0))

res = pd.DataFrame(rows, columns=[
    "lag_quarters","n","auc","mean_not_supp","mean_supp","diff_supp_minus_not"
]).set_index("lag_quarters")

print("\n=== SoCal Test 1 Results ===")
print(res)

best_lag = res["auc"].idxmax()
print(f"\nBEST LAG = {best_lag} quarters  (AUC={res.loc[best_lag,'auc']:.3f}, n={int(res.loc[best_lag,'n'])})")

# Save plots to folder
FIG_DIR = Path("../../5_FIGURES/so cal test 1")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Plot AUC vs lag
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(res.index, res["auc"], marker="o")
ax.axhline(0.5, linestyle="--")
ax.set_title("SoCal: Predict suppressed using SST anomaly max (AUC vs lag)")
ax.set_xlabel("Lag (quarters) — SST earlier → suppressed later")
ax.set_ylabel("AUC")
fig.tight_layout()
out1 = FIG_DIR / "socal_auc_suppressed_sstanom_q_max.png"
fig.savefig(out1, dpi=200, bbox_inches="tight")
plt.close(fig)
print("saved:", out1.resolve())

# Overlay plot using best lag
x_best = x0.shift(best_lag)
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(d.index, x_best, label=f"sstanom_q_max (lag {best_lag}q)")
ax.plot(d.index, y, label="suppressed (0/1)", alpha=0.8)
ax.set_title(f"SoCal: suppressed vs lagged SST anomaly peaks (lag={best_lag} quarters)")
ax.set_xlabel("Year (quarter start)")
ax.set_ylabel("SST anomaly (°C) / label")
ax.legend()
fig.tight_layout()
out2 = FIG_DIR / f"socal_overlay_suppressed_vs_sstanom_q_max_lag{best_lag}.png"
fig.savefig(out2, dpi=200, bbox_inches="tight")
plt.close(fig)
print("saved:", out2.resolve())

# Boxplot at best lag
tmp = pd.DataFrame({"x": x_best, "y": y}).dropna()
g0 = tmp[tmp.y == 0].x.values
g1 = tmp[tmp.y == 1].x.values

fig, ax = plt.subplots(figsize=(6,4))
ax.boxplot([g0, g1], tick_labels=["not suppressed", "suppressed"])
ax.set_title(f"SoCal: SST anomaly peak (lag {best_lag}q) by kelp state")
ax.set_ylabel("sstanom_q_max (°C)")
fig.tight_layout()
out3 = FIG_DIR / f"socal_box_sstanom_q_max_lag{best_lag}_by_suppressed.png"
fig.savefig(out3, dpi=200, bbox_inches="tight")
plt.close(fig)
print("saved:", out3.resolve())

Loaded: /Users/tonylin/Documents/kelp_project/1_DATA/processed/socal_kelp_sst_labeled.csv
Rows: 167 Time: 1984-01-01 00:00:00 -> 2025-07-01 00:00:00
Suppressed counts: {0: 159, 1: 8}

=== SoCal Test 1 Results ===
                n       auc  mean_not_supp  mean_supp  diff_supp_minus_not
lag_quarters                                                              
0             163  0.346774       0.470886   0.171002            -0.299884
1             162  0.457792       0.451165   0.422385            -0.028780
2             161  0.595588       0.434818   0.735351             0.300533
3             160  0.724556       0.428678   1.050633             0.621955
4             159  0.669935       0.448464   0.772959             0.324495
5             158  0.705882       0.452828   0.855541             0.402713
6             157  0.514706       0.469693   0.413281            -0.056411
7             156  0.478618       0.469748   0.333960            -0.135788
8             155  0.354305       0.4

In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

d = pd.read_csv("../../1_DATA/processed/socal_kelp_sst_labeled.csv",
                index_col=0, parse_dates=True).sort_index()

# align to quarter start
d.index = d.index.to_period("Q").to_timestamp(how="start")

lag = 4
x = d["sstanom_q_max"].shift(lag)          # SST peak earlier by 4 quarters
y = d["suppressed"].astype(int)

tmp = pd.DataFrame({"x": x, "y": y}).dropna()

g0 = tmp[tmp.y == 0].x.values
g1 = tmp[tmp.y == 1].x.values

auc = roc_auc_score(tmp.y, tmp.x)

mean0, mean1 = g0.mean(), g1.mean()
diff = mean1 - mean0
pooled = np.sqrt(((len(g0)-1)*g0.var(ddof=1) + (len(g1)-1)*g1.var(ddof=1)) / (len(g0)+len(g1)-2))
d_cohen = diff / pooled

print("lag:", lag, "quarters")
print("n not suppressed:", len(g0), "mean:", mean0)
print("n suppressed    :", len(g1), "mean:", mean1)
print("mean diff (supp - not):", diff)
print("Cohen's d:", d_cohen)
print("AUC:", auc)

lag: 4 quarters
n not suppressed: 153 mean: 0.44846358683660137
n suppressed    : 6 mean: 0.7729589116666667
mean diff (supp - not): 0.3244953248300654
Cohen's d: 0.3646221649077204
AUC: 0.6699346405228759


In [5]:
import pandas as pd

# ERDDAP CSV has a 2nd row of units -> skiprows=[1]
url = (
    "https://coastwatch.pfeg.noaa.gov/erddap/griddap/erdUI39mo.csv"
    "?upwelling_index[(1984-01-15T00:00:00Z):1:(2025-12-15T00:00:00Z)][(39.0)][(235.0)],"
    "upwelling_index_anomaly[(1984-01-15T00:00:00Z):1:(2025-12-15T00:00:00Z)][(39.0)][(235.0)]"
)

ui = pd.read_csv(url, skiprows=[1])
ui["time"] = pd.to_datetime(ui["time"])
ui = ui.set_index("time").sort_index()

# align monthly timestamps to month-start for easier joins
ui.index = ui.index.to_period("M").to_timestamp(how="start")

ui_q = ui.resample("QS").agg({
    "upwelling_index": "mean",
    "upwelling_index_anomaly": "mean",
})
ui_q["upwelling_index_lag1"] = ui_q["upwelling_index"].shift(1)

print(ui_q.head())

            upwelling_index  upwelling_index_anomaly  upwelling_index_lag1
time                                                                      
1984-01-01        15.333333                 4.666667                   NaN
1984-04-01       197.333333                77.333333             15.333333
1984-07-01       213.333333                85.333333            197.333333
1984-10-01         2.333333                 2.000000            213.333333
1985-01-01        43.333333                32.666667              2.333333


/var/folders/61/3wm_gk5j5jvd7cv3mv31_8940000gn/T/ipykernel_55165/908861498.py:15: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  ui.index = ui.index.to_period("M").to_timestamp(how="start")
